# This is the Make_sum_more model using Multi-layer perceptron.

In [132]:
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline


In [133]:
#Read all of the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [134]:
len(words)

32033

In [135]:
#Build the vocab of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [136]:
#Build the dataset

block_size = 3 #context length, how many characters should we take to predict the next one
X, y = [], []
for w in words:
    # print(w)
    context = [0] * block_size
    for ch in w + '.': #for padding
        ix = stoi[ch]
        X.append(context)
        y.append(ix)
        # print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] #crop and append

X = torch.tensor(X)
y = torch.tensor(y)

In [137]:
X.shape, X.dtype, y.shape, y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [138]:
C = torch.randn((27, 2)) #(rows, columns)

In [139]:
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

In [140]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [141]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [142]:
h

tensor([[ 0.9403, -0.8648, -0.6827,  ...,  0.9266,  0.9298,  0.6440],
        [ 0.9317, -0.9214, -0.8919,  ...,  0.8940, -0.3752,  0.8769],
        [-0.0116,  0.7532, -0.9873,  ...,  0.9325,  0.9999, -0.6810],
        ...,
        [ 0.9399, -0.6277, -0.9986,  ...,  0.4244, -0.6927,  0.8943],
        [ 0.9355,  0.1403, -0.9995,  ...,  0.3008,  0.9965, -0.3468],
        [ 0.9998, -0.3625,  0.3379,  ...,  0.9833, -0.1941,  0.5512]])

In [143]:
h.shape

torch.Size([228146, 100])

In [144]:
W2 = torch.randn(100, 27)
b2 = torch.randn(27)

In [145]:
logits = h @ W2 + b2

In [146]:
counts = logits.exp()

In [147]:
prob = counts / counts.sum(1, keepdims=True)

In [148]:
prob[0].sum() #adds to one, so it normalized

tensor(1.)

In [149]:
y

tensor([ 5, 13, 13,  ..., 26, 24,  0])

In [150]:
loss = -prob[torch.arange(32), y].log().mean()
loss

IndexError: shape mismatch: indexing tensors could not be broadcast together with shapes [32], [228146]

In [151]:
F.cross_entropy(logits, y)
parameters = [C, W1, b1, W2, b2]

In [157]:
g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((27, 2))
W1 = torch.randn((6, 100))
b1 = torch.randn(100)
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

parameters = [C, W1, b1, W2, b2]
for p in parameters:
    p.requires_grad = True

In [162]:
#training loop
epochs = 10
for epoch in range(epochs):
    #mini-batch constuct
    ix = torch.randint(0, X.shape[0], (32,))

    #forward pass
    emb = C[X[ix]] #[32, 3, 2]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1) #[32, 100]
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y[ix])
    print(loss.item())
    #Backpropogation
    for p in parameters:
        p.grad = None
    loss.backward()
    #update
    for p in parameters:
        p.data += -0.1 * p.grad #0.1 is the lr (learning rate)



2.8489551544189453
2.6913275718688965
3.4316020011901855
2.777613878250122
2.432594060897827
2.9113152027130127
3.221736192703247
2.9830477237701416
3.509857654571533
3.2692525386810303
